# Du fichier IMDb au catalogue final

Ce notebook fait **toute la démarche**, du début à la fin :

1. lire les données **IMDb** (en ligne) ;
2. filtrer les films (nos règles : 1970, 60 min, 10 000 votes, note 6, 7 genres) ;
3. ajouter les **titres français** (`title.akas`) ;
4. lire la base **TMDB** (le fichier CSV du cours) et la **relier** à IMDb ;
5. repérer les **films français** ;
6. contrôler qu'il n'y a **aucun doublon** ;
7. enregistrer `data/catalogue_final.csv`, qui servira au tableau de bord et à l'appli.

## Où sont les données ?

| Base | Où la trouver | Comment on la lit |
| --- | --- | --- |
| IMDb | En ligne sur https://datasets.imdbws.com (fichiers `.tsv.gz`) | Directement avec l'adresse internet, sans rien télécharger |
| TMDB | Sur ton PC : `PROJET_02/Docs/TMDB_movie_dataset_v11.csv` (674 Mo) | Avec le chemin du fichier sur le disque |

## Comment on relie les deux bases ?

Chaque film a un **identifiant IMDb** unique, par exemple `tt0076759` pour *Star Wars*.
Il s'appelle `tconst` dans IMDb, `titleId` dans `title.akas` et `imdb_id` dans TMDB.
C'est cette **clé** qui permet de coller les informations des deux bases sur la même ligne.

**Avant de commencer :** il faut une connexion internet. Lance les cellules **dans l'ordre**, de haut en bas. Le notebook entier prend plusieurs minutes (la lecture de `title.akas` est la plus longue).

# Guide : les tâches et les notions

Ce guide reprend le document « Projet 2 — Les notions data et comment les appliquer ». Les **tâches** sont ci-dessous. Les **14 notions** expliquées en détail sont à la **fin du notebook**, dans l'annexe. Clique sur « notion N » pour y aller.

Dans les exemples des notions, `films` est ton tableau de films.

## Les tâches à faire

**Pour le point client (ven. 20/11, 14h)**

- [ ] Lire `title.akas` par morceaux et garder la région FR. Voir [notion 1](#notion1).
- [ ] Enlever les doublons de titres. Voir [notion 5](#notion5).
- [ ] Joindre les titres FR au catalogue (`left`). Voir [notion 4](#notion4).
- [ ] Remplacer les titres FR manquants par le titre original. Voir [notion 6](#notion6).
- [ ] Compléter la slide 5 avec la méthode et les chiffres.

**Pour le catalogue final (lun. 30/11)**

- [ ] Joindre TMDB (résumé, affiche, pays). Voir [notion 4](#notion4).
- [ ] Repérer les films français. Voir [notion 7](#notion7).
- [ ] Vérifier qu'il n'y a aucun doublon, puis enregistrer `catalogue_final.csv`. Voir [notion 5](#notion5).

**Pour le ven. 04/12**

- [ ] Calculer les KPI du tableau de bord. Voir [notion 8](#notion8).
- [ ] Classer les films à mettre en avant. Voir [notion 9](#notion9).
- [ ] Construire le tableau de bord. Voir [notion 10](#notion10).
- [ ] Préparer les données pour la recommandation. Voir [notion 11](#notion11).
- [ ] Coder et tester la recommandation. Voir [notion 12](#notion12).
- [ ] Créer l'appli Streamlit et la faire tester par une autre équipe. Voir [notion 13](#notion13).

**Pour la soutenance (lun. 07/12)**

- [ ] Finir les slides et mettre le dépôt GitHub à jour. Voir [notion 14](#notion14).

Les tâches du point client et du catalogue final sont **toutes faites par ce notebook** (parties 1 à 5). Il te reste à noter les chiffres pour les slides.

In [ ]:
import pandas as pd
from pathlib import Path

racine = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
dossier_data = racine / "data"
fichier_tmdb = racine.parent / "Docs" / "TMDB_movie_dataset_v11.csv"
dossier_data.mkdir(exist_ok=True)

print("Dossier du projet :", racine)
print("Fichier TMDB trouvé :", fichier_tmdb.exists())

### Explication : préparer les chemins

- `import pandas as pd` charge **pandas**, la bibliothèque qui manipule les tableaux.
- `Path` (bibliothèque `pathlib`) sert à écrire des chemins de fichiers qui marchent sous Windows.
- `racine` est le dossier `projet2-cinema`. Si le notebook tourne depuis le dossier `notebooks`, on remonte d'un cran avec `.parent`.
- `dossier_data` est le dossier `projet2-cinema/data`, où on enregistre nos fichiers. `mkdir(exist_ok=True)` le crée s'il n'existe pas.
- `fichier_tmdb` pointe vers `PROJET_02/Docs/TMDB_movie_dataset_v11.csv`. On remonte deux fois : `projet2-cinema` puis `PROJET_02`.

**À vérifier :** la dernière ligne doit afficher `Fichier TMDB trouvé : True`. Si elle affiche `False`, le fichier n'est pas au bon endroit : corrige le chemin de `fichier_tmdb`.

---
# Partie 1 : lire IMDb et filtrer les films

In [ ]:
url_ratings = "https://datasets.imdbws.com/title.ratings.tsv.gz"
url_basics = "https://datasets.imdbws.com/title.basics.tsv.gz"
url_akas = "https://datasets.imdbws.com/title.akas.tsv.gz"

ratings = pd.read_csv(url_ratings, sep="\t", na_values="\\N")

colonnes_basics = ["tconst", "titleType", "primaryTitle", "originalTitle", "isAdult",
                   "startYear", "runtimeMinutes", "genres"]
basics = pd.read_csv(url_basics, sep="\t", na_values="\\N",
                     usecols=colonnes_basics, low_memory=False)

print("basics :", basics.shape)
print("ratings :", ratings.shape)

### Explication : lire IMDb en ligne

- On range les **adresses** des 3 fichiers IMDb dans des variables. On s'en servira plus bas.
- `pd.read_csv(adresse, ...)` lit le fichier directement depuis internet.
- `sep="\t"` : les colonnes sont séparées par des **tabulations** (fichiers `.tsv`).
- `na_values="\\N"` : IMDb écrit `\N` quand une information manque. pandas les remplace par une case vide (`NaN`).
- `usecols=colonnes_basics` : on ne charge que les colonnes utiles. `title.basics` fait environ 13 millions de lignes, cela économise beaucoup de mémoire.
- `.shape` affiche (nombre de lignes, nombre de colonnes).

**Ce qu'on doit voir :** environ 13 millions de lignes pour `basics` et environ 1,6 million pour `ratings`.

In [ ]:
basics["startYear"] = pd.to_numeric(basics["startYear"], errors="coerce")
basics["runtimeMinutes"] = pd.to_numeric(basics["runtimeMinutes"], errors="coerce")
basics["isAdult"] = pd.to_numeric(basics["isAdult"], errors="coerce")

print(basics.dtypes)
print(basics.isna().sum())

### Explication : mettre les bons types

- L'année, la durée et `isAdult` sont lues comme du **texte**. On ne peut pas écrire `>= 1970` sur du texte.
- `pd.to_numeric(..., errors="coerce")` les transforme en **nombres**. Une valeur illisible devient vide au lieu de faire planter le code.
- `dtypes` affiche le type de chaque colonne : on doit voir `float64` pour les trois colonnes converties.
- `isna().sum()` compte les cases vides par colonne. C'est le contrôle demandé après chaque lecture.

In [ ]:
etapes = []
films = basics.copy()
etapes.append(["Tous les titres IMDb", len(films)])

films = films[films["titleType"] == "movie"]
etapes.append(["Films de cinéma", len(films)])

films = films[films["isAdult"] == 0]
etapes.append(["Sans films adultes", len(films)])

films = films[films["startYear"] >= 1970]
etapes.append(["Depuis 1970", len(films)])

films = films[films["runtimeMinutes"] >= 60]
etapes.append(["60 minutes ou plus", len(films)])

films = films.merge(ratings, on="tconst", how="inner")
etapes.append(["Films avec une note", len(films)])

films = films[films["numVotes"] >= 10000]
etapes.append(["10 000 votes ou plus", len(films)])

films = films[films["averageRating"] >= 6]
etapes.append(["Note de 6 ou plus", len(films)])

films = films[films["genres"].str.contains("Action|Horror|Thriller|Sci-Fi|Adventure|War|Comedy", na=False)]
etapes.append(["Nos 7 genres", len(films)])

pd.DataFrame(etapes, columns=["Étape", "Nombre de films"])

### Explication : l'entonnoir des filtres

C'est la même démarche que dans `preparation_donnees.ipynb`. On applique un critère à la fois et on note le nombre de films restants (`len(films)`) dans la liste `etapes`.

- `films[condition]` garde seulement les lignes où la condition est vraie.
- `titleType == "movie"` : uniquement les films de cinéma (pas les séries ni les épisodes).
- `isAdult == 0` : on retire les films pour adultes.
- `startYear >= 1970` et `runtimeMinutes >= 60` : films sortis depuis 1970, d'au moins une heure.
- `merge(ratings, on="tconst", how="inner")` **colle les notes** à chaque film grâce à la clé `tconst`. Avec `inner`, les films sans note disparaissent.
- `numVotes >= 10000` (films connus) et `averageRating >= 6` (films bien notés).
- `str.contains("Action|Horror|...")` garde les films qui ont **au moins un** des 7 genres. Le `|` veut dire « ou ».

**Ce qu'on doit voir :** environ 6 447 films à la fin (le chiffre peut bouger un peu, car IMDb met ses fichiers à jour chaque jour).

---
# Partie 2 : ajouter les titres français (`title.akas`)

In [ ]:
ids_catalogue = set(films["tconst"])
morceaux = []

for morceau in pd.read_csv(url_akas, sep="\t", na_values="\\N",
                           usecols=["titleId", "title", "region", "types"],
                           chunksize=2_000_000):
    morceau = morceau[(morceau["region"] == "FR") & (morceau["titleId"].isin(ids_catalogue))]
    morceaux.append(morceau)

akas_fr = pd.concat(morceaux)
print(len(akas_fr), "titres FR trouvés pour nos films")
akas_fr.head(10)

### Explication : lire `title.akas` par morceaux

`title.akas` contient **tous les titres de tous les films dans tous les pays** : environ 60 millions de lignes. C'est trop gros pour la mémoire d'un PC. On le lit donc **par morceaux**.

- `ids_catalogue` est l'ensemble des identifiants de nos films. Un `set` permet de chercher dedans très vite.
- `chunksize=2_000_000` : pandas ne lit pas tout d'un coup. Il donne le fichier en morceaux de 2 millions de lignes, l'un après l'autre, dans la boucle `for`.
- Dans chaque morceau, on garde seulement les lignes où :
    - `region == "FR"` : le titre utilisé **en France** ;
    - `titleId` fait partie de nos films.
- `morceaux.append(...)` met de côté ce petit morceau filtré.
- `pd.concat(morceaux)` recolle tous les petits morceaux en un seul tableau.

**Attention :** on ne filtre **pas** sur `language == "fr"`. Cela donnerait surtout des titres québécois.

Cette cellule est la plus longue du notebook (plusieurs minutes). C'est normal.

In [ ]:
print("Lignes avant :", len(akas_fr))
print("Films en double :", akas_fr["titleId"].duplicated().sum())

akas_fr["priorite"] = (akas_fr["types"] == "imdbDisplay").astype(int)
akas_fr = akas_fr.sort_values("priorite", ascending=False)
akas_fr = akas_fr.drop_duplicates("titleId")

print("Lignes après :", len(akas_fr))

### Explication : un seul titre FR par film

Un même film peut avoir **plusieurs titres** pour la France (titre de sortie en salle, titre vidéo, titre télé...). Si on les garde tous, le film apparaîtra plusieurs fois dans le catalogue après la jointure. Il faut donc en garder **un seul**.

- `duplicated().sum()` compte les lignes dont l'identifiant a déjà été vu plus haut : ce sont les doublons.
- La colonne `types` indique la nature du titre. `imdbDisplay` est le titre qu'IMDb affiche en France : c'est le meilleur choix.
- `priorite` vaut 1 pour un titre `imdbDisplay` et 0 sinon. `astype(int)` transforme Vrai/Faux en 1/0.
- `sort_values("priorite", ascending=False)` met les titres `imdbDisplay` en premier.
- `drop_duplicates("titleId")` garde la **première** ligne de chaque film et supprime les autres.

**Ce qu'on doit voir :** « Lignes après » = nombre de films qui ont un titre FR. Note ce chiffre pour la slide 5.

In [ ]:
akas_fr = akas_fr.rename(columns={"titleId": "tconst", "title": "titre_fr"})

avant = len(films)
films = films.merge(akas_fr[["tconst", "titre_fr"]], on="tconst", how="left")
print("Films avant :", avant, "| après :", len(films))

sans_titre = films["titre_fr"].isna().sum()
print(sans_titre, "films sans titre français")

films["titre_fr"] = films["titre_fr"].fillna(films["primaryTitle"])

### Explication : coller les titres FR au catalogue

- `rename(...)` change le nom des colonnes : `titleId` devient `tconst` (le même nom que dans `films`) et `title` devient `titre_fr`, plus clair.
- `merge(..., on="tconst", how="left")` colle le titre français à côté de chaque film.
    - `how="left"` garde **tous nos films**, même ceux qui n'ont pas de titre FR. Leur case `titre_fr` reste vide.
    - Avec `how="inner"`, on perdrait ces films : ce n'est pas ce qu'on veut.
- On affiche le nombre de films avant et après : **il doit être identique**. S'il augmente, il reste des doublons dans `akas_fr`.
- `isna().sum()` compte les films sans titre FR.
- **Règle de repli :** `fillna(films["primaryTitle"])` remplit chaque case vide avec le titre original du même film. Ainsi, chaque film a un titre à afficher.

In [ ]:
exemples = ["tt0073195", "tt0078748", "tt0076759", "tt0361748", "tt0077651"]
films[films["tconst"].isin(exemples)][["tconst", "primaryTitle", "titre_fr"]]

### Explication : vérifier avec des films connus

On affiche 5 films célèbres pour vérifier que les titres français sont les bons : *Jaws* doit devenir *Les Dents de la mer*, *Star Wars* doit devenir *La Guerre des étoiles*.

- `isin(exemples)` garde les films dont l'identifiant est dans la liste.
- `[["tconst", "primaryTitle", "titre_fr"]]` n'affiche que ces 3 colonnes.

Ces exemples sont parfaits pour la slide 5 (« titre original → titre FR »).

---
# Partie 3 : relier IMDb et TMDB

In [ ]:
apercu = pd.read_csv(fichier_tmdb, nrows=5)
print(list(apercu.columns))
apercu.head()

### Explication : regarder TMDB avant de le charger

Le fichier TMDB pèse 674 Mo. Avant de le charger en entier, on regarde à quoi il ressemble.

- `nrows=5` ne lit que les **5 premières lignes** : c'est instantané.
- `list(apercu.columns)` affiche le nom de toutes les colonnes.

**À vérifier :** on doit voir `imdb_id` (la clé pour relier à IMDb), `overview` (le résumé), `poster_path` (l'affiche), `production_countries` (les pays de production) et `original_language`. Si un nom est différent, corrige-le dans la cellule suivante.

In [ ]:
colonnes_tmdb = ["imdb_id", "vote_count", "overview", "poster_path",
                 "production_countries", "original_language"]
tmdb = pd.read_csv(fichier_tmdb, usecols=colonnes_tmdb)
print("TMDB brut :", tmdb.shape)

tmdb = tmdb.dropna(subset=["imdb_id"])
print("Avec un imdb_id :", len(tmdb))
print("imdb_id en double :", tmdb["imdb_id"].duplicated().sum())

tmdb = tmdb.sort_values("vote_count", ascending=False)
tmdb = tmdb.drop_duplicates("imdb_id")
tmdb = tmdb.drop(columns="vote_count")
print("Après dédoublonnage :", len(tmdb))

### Explication : charger et nettoyer TMDB

- `usecols=colonnes_tmdb` : on ne charge que les 6 colonnes utiles, pour ne pas saturer la mémoire.
- `dropna(subset=["imdb_id"])` supprime les films TMDB qui n'ont **pas d'identifiant IMDb**. On ne pourrait de toute façon pas les relier.
- `duplicated().sum()` compte les `imdb_id` présents plusieurs fois. Il y en a quelques-uns dans TMDB.
- Pour choisir quelle fiche garder, on trie par `vote_count` (nombre de votes TMDB), du plus grand au plus petit. La fiche la plus votée est en général la bonne.
- `drop_duplicates("imdb_id")` garde la première ligne de chaque film, donc la plus votée.
- `drop(columns="vote_count")` supprime cette colonne : elle ne servait qu'au tri.

**Pourquoi c'est important :** si un `imdb_id` apparaît 2 fois dans TMDB, le film apparaîtra 2 fois dans notre catalogue après la jointure.

In [ ]:
avant = len(films)
films = films.merge(tmdb, left_on="tconst", right_on="imdb_id", how="left")
print("Films avant :", avant, "| après :", len(films))

trouves = films["imdb_id"].notna().sum()
print(trouves, "films sur", len(films), "trouvés dans TMDB")

films = films.drop(columns="imdb_id")

### Explication : la jointure IMDb + TMDB

C'est l'étape qui **relie les deux bases**.

- `merge(tmdb, left_on="tconst", right_on="imdb_id", how="left")` :
    - `left_on="tconst"` : la clé du côté de nos films (IMDb) ;
    - `right_on="imdb_id"` : la clé du côté de TMDB ;
    - pandas colle la ligne TMDB qui a le même identifiant à côté de chaque film ;
    - `how="left"` garde **tous nos films**, même ceux qui ne sont pas dans TMDB (leurs colonnes TMDB restent vides).
- Le nombre de films avant et après doit être **identique**.
- `notna().sum()` compte les films qui ont trouvé une fiche TMDB. On attend environ 97 % (6 255 sur 6 447 dans le premier notebook).
- `drop(columns="imdb_id")` supprime la colonne `imdb_id` : elle est identique à `tconst`, on n'en a plus besoin.

---
# Partie 4 : repérer les films français

In [ ]:
films["film_francais"] = films["production_countries"].str.contains("France", na=False)

nb_francais = films["film_francais"].sum()
part = films["film_francais"].mean() * 100
print(nb_francais, "films français, soit", round(part, 1), "% du catalogue")

films[films["film_francais"]][["titre_fr", "startYear", "production_countries"]].head(10)

### Explication : qu'est-ce qu'un film français ?

**Règle choisie :** un film est français si la colonne `production_countries` de TMDB contient « France ».

- `str.contains("France", na=False)` renvoie Vrai si le texte contient le mot France. `na=False` : si la case est vide (film absent de TMDB), on compte Faux.
- Le résultat est rangé dans une nouvelle colonne `film_francais` (Vrai / Faux).
- `sum()` compte les Vrai (Vrai vaut 1, Faux vaut 0) : c'est le nombre de films français.
- `mean() * 100` donne la **part** de films français en pourcentage.
- La dernière ligne affiche 10 films français pour vérifier que la règle a du sens.

**La limite à écrire dans les slides :** une **coproduction** compte comme française (par exemple un film américain coproduit par la France).
L'autre règle possible, `original_language == "fr"`, compterait aussi les films québécois, belges et suisses, et raterait des films français tournés en anglais comme *Lucy* ou *Le Cinquième Élément*.

---
# Partie 5 : contrôles et enregistrement

In [ ]:
print("Une ligne par film :", films["tconst"].is_unique)
print("Nombre de films :", len(films))
print()
print(films.isna().sum())

### Explication : le contrôle final

C'est un **critère d'évaluation** du projet : une ligne par film, sans doublon.

- `films["tconst"].is_unique` doit afficher **True**. Cela veut dire que chaque identifiant n'apparaît qu'une seule fois.
- Si ça affiche **False**, une jointure a créé des doublons. Pour les voir : `films[films["tconst"].duplicated(keep=False)]`.
- `isna().sum()` montre combien de cases sont vides dans chaque colonne. Il est normal d'avoir des vides dans `overview`, `poster_path` et `production_countries` pour les films absents de TMDB. Note ces chiffres pour les slides (partie « difficultés »).

In [ ]:
films["affiche"] = "https://image.tmdb.org/t/p/w500" + films["poster_path"]

colonnes_finales = ["tconst", "titre_fr", "primaryTitle", "originalTitle", "startYear",
                    "runtimeMinutes", "genres", "averageRating", "numVotes",
                    "film_francais", "production_countries", "original_language",
                    "overview", "poster_path", "affiche"]
catalogue = films[colonnes_finales]

catalogue.to_csv(dossier_data / "catalogue_final.csv", index=False)
print(len(catalogue), "films enregistrés dans data/catalogue_final.csv")
catalogue.head()

### Explication : enregistrer le catalogue final

- `affiche` : on construit l'adresse complète de l'affiche en collant le début fixe `https://image.tmdb.org/t/p/w500` et le `poster_path` du film. Pas besoin de clé d'API. Si `poster_path` est vide, `affiche` est vide aussi.
- `colonnes_finales` choisit les colonnes à garder, dans un ordre lisible.
- `to_csv(..., index=False)` enregistre le tableau dans `data/catalogue_final.csv`. `index=False` évite une colonne inutile de numéros de ligne.

Ce fichier sert ensuite :
- au **tableau de bord** (Power BI ou autre) ;
- au **notebook de recommandation** ;
- à l'**appli Streamlit**.

**Attention :** ce fichier ne doit pas aller sur GitHub si le dossier `data/` est dans le `.gitignore`. Partage-le à l'équipe par le Drive.

In [ ]:
etapes.append(["Avec un titre français (région FR)", len(films) - sans_titre])
etapes.append(["Trouvés dans TMDB", trouves])
etapes.append(["Films français (production_countries)", nb_francais])

bilan = pd.DataFrame(etapes, columns=["Étape", "Nombre de films"])
bilan.to_csv(dossier_data / "bilan_catalogue_final.csv", index=False)
bilan

### Explication : le bilan pour les slides

On ajoute à la liste `etapes` les chiffres des parties 2 à 4 :
- le nombre de films qui ont un vrai titre français ;
- le nombre de films trouvés dans TMDB ;
- le nombre de films français.

Le tableau est enregistré dans `data/bilan_catalogue_final.csv`. Il contient **tous les chiffres avant/après** demandés par la commission. Tu peux le recopier dans les slides.

---
# Annexe : les 14 notions data et comment les appliquer

Pour chaque notion : ce que c'est, pourquoi on en a besoin, et le code à écrire dans ton projet. Les notions 1 à 3 sont déjà dans `preparation_donnees.ipynb`. Les notions 4 à 7 sont utilisées dans ce notebook. Les notions 8 à 14 servent pour le tableau de bord, la recommandation et l'appli.

<a id="notion1"></a>
## Notions 1 à 3 : lire, typer, filtrer

**1. Lire un fichier.** `pd.read_csv` transforme un fichier en tableau (DataFrame). Les fichiers IMDb sont séparés par des tabulations (`sep="\t"`) et écrivent les cases vides `\N` (`na_values="\\N"`). `usecols` ne charge que les colonnes utiles, pour économiser la mémoire.

Ce qui est nouveau pour toi : `chunksize`. Certains fichiers sont trop gros pour la mémoire de ton PC (`title.akas` fait environ 60 millions de lignes). Avec `chunksize=2_000_000`, pandas ne lit pas tout d'un coup. Il te donne le fichier en morceaux de 2 millions de lignes, l'un après l'autre, dans une boucle `for`. Dans chaque morceau, tu gardes seulement ce qui t'intéresse, puis tu recolles les petits morceaux avec `pd.concat`.

```python
url_akas = "https://datasets.imdbws.com/title.akas.tsv.gz"
morceaux = []
for m in pd.read_csv(url_akas, sep="\t", na_values="\\N",
                     usecols=["titleId", "title", "region"],
                     chunksize=2_000_000):
    m = m[(m["region"] == "FR") & (m["titleId"].isin(films["tconst"]))]
    morceaux.append(m)
akas_fr = pd.concat(morceaux)
print(akas_fr.shape)
```

**2. Les types de colonnes.** Une année lue comme du texte (`"1977"`) ne peut pas être comparée à 1970. `pd.to_numeric(..., errors="coerce")` la transforme en nombre. Après chaque lecture, vérifie `df.dtypes`, `df.shape` et `df.isna().sum()`.

**3. Filtrer.** `films[films["startYear"] >= 1970]` garde les lignes où la condition est vraie. On combine deux conditions avec `&` (et) ou `|` (ou), chacune entre parenthèses. Tu l'as fait en entonnoir, avec `len(films)` après chaque filtre : c'est exactement ce que la commission attend.

<a id="notion4"></a>
## Notion 4 : les jointures (merge)

Une jointure colle deux tableaux côte à côte, ligne par ligne, grâce à une colonne qu'ils ont en commun : la **clé**. Dans ton projet, la clé est toujours l'identifiant IMDb du film (`tt0076759` pour Star Wars). Elle s'appelle `tconst` dans IMDb, `titleId` dans `title.akas` et `imdb_id` dans TMDB.

Imagine deux listes : la tienne, avec tes 6 447 films, et une autre avec des titres français. La jointure cherche, pour chacun de tes films, la ligne de l'autre liste qui a le même identifiant, et recopie ses colonnes à côté.

La question importante est : que fait-on des films qui n'ont pas de correspondance ? C'est le rôle du paramètre `how`.

| `how=` | Ce qui reste | Quand l'utiliser dans ton projet |
| --- | --- | --- |
| `"inner"` | Seulement les films présents dans les deux tableaux | Les notes (`title.ratings`) : un film sans note ne nous intéresse pas. Tu l'as déjà fait. |
| `"left"` | Tous les films du tableau de gauche, même sans correspondance (les cases manquantes deviennent vides, `NaN`) | Les titres français et TMDB : on ne veut perdre aucun film juste parce qu'il n'a pas de titre FR ou pas de fiche TMDB. |

Le « tableau de gauche » est celui sur lequel tu appelles `.merge(...)`, donc `films`.

**Exemple 1 : ajouter les titres français.** Les noms de colonnes ne sont pas les mêmes, donc on renomme d'abord `titleId` en `tconst`, et `title` en `titre_fr` pour que ce soit clair.

```python
akas_fr = akas_fr.rename(columns={"titleId": "tconst", "title": "titre_fr"})
akas_fr = akas_fr.drop_duplicates("tconst")
films = films.merge(akas_fr[["tconst", "titre_fr"]], on="tconst", how="left")
print(len(films))
```

**Exemple 2 : ajouter TMDB.** Ici, on ne renomme pas : on dit à pandas quelle colonne prendre de chaque côté avec `left_on` et `right_on`.

```python
colonnes_tmdb = ["imdb_id", "overview", "poster_path",
                 "production_countries", "original_language"]
tmdb = pd.read_csv(fichier_tmdb, usecols=colonnes_tmdb)
tmdb = tmdb.drop_duplicates("imdb_id")
films = films.merge(tmdb, left_on="tconst", right_on="imdb_id", how="left")
```

**Le réflexe à prendre :** affiche `len(films)` avant et après chaque jointure. Avec `how="left"`, le nombre doit rester exactement le même. S'il augmente, c'est qu'il y avait des doublons dans l'autre tableau.

<a id="notion5"></a>
## Notion 5 : les doublons

Un doublon, c'est une même clé qui apparaît sur plusieurs lignes. Le cahier des charges est clair : le catalogue final doit avoir **une ligne par film**, sans doublon. C'est un critère d'évaluation.

**Pourquoi c'est un piège avec les jointures.** Dans `title.akas`, un même film peut avoir plusieurs titres pour la France (titre de sortie, titre vidéo, titre télé...). Si *Alien* a 3 lignes FR et que tu fais la jointure sans précaution, *Alien* apparaît 3 fois dans ton catalogue. Pareil si TMDB contient deux fiches pour le même `imdb_id`.

**La règle : on nettoie l'autre tableau avant de joindre.**

```python
print(akas_fr["titleId"].duplicated().sum())
akas_fr = akas_fr.drop_duplicates("titleId")
```

- `duplicated()` renvoie Vrai pour chaque ligne dont la clé a déjà été vue plus haut. `.sum()` les compte.
- `drop_duplicates("titleId")` supprime ces lignes et garde la première. Note ce chiffre : c'est une bonne info pour la slide 5.

**La vérification finale, après toutes les jointures :**

```python
print(films["tconst"].is_unique)
```

Elle doit afficher `True`. Si ça affiche `False`, cherche quels films posent problème avec `films[films["tconst"].duplicated(keep=False)]`, qui affiche toutes les lignes en double.

<a id="notion6"></a>
## Notion 6 : les valeurs manquantes

Une valeur manquante (`NaN`) est une case vide. Après une jointure `left`, chaque film qui n'a pas trouvé de correspondance a des cases vides dans les nouvelles colonnes. Il faut décider quoi en faire, et l'écrire dans les slides.

| Outil | Ce qu'il fait | Exemple dans ton projet |
| --- | --- | --- |
| `isna()` | Repère les cases vides (Vrai / Faux) | Compter les films sans titre FR : `films["titre_fr"].isna().sum()` |
| `fillna(valeur)` | Remplit les cases vides | Mettre le titre original quand il n'y a pas de titre FR |
| `dropna(subset=[...])` | Supprime les lignes qui ont une case vide | Retirer de l'appli les films sans affiche, si l'équipe le décide |

**La règle de repli des titres** (c'est le point 03 à compléter sur la slide 5) :

```python
sans_titre = films["titre_fr"].isna().sum()
print(sans_titre, "films sans titre français")
films["titre_fr"] = films["titre_fr"].fillna(films["primaryTitle"])
```

Avec `fillna(films["primaryTitle"])`, pandas remplit chaque case vide avec le titre original de la même ligne. Garde le chiffre `sans_titre` pour la slide.

**Attention :** ne supprime pas trop vite avec `dropna()`. Un film sans résumé reste un bon film du catalogue. Supprime seulement quand la case vide empêche vraiment l'usage, et compte combien de films tu retires.

<a id="notion7"></a>
## Notion 7 : travailler sur du texte

Plusieurs colonnes contiennent plusieurs informations dans une seule case, séparées par des virgules. Par exemple, `genres` vaut `"Action,Adventure,Sci-Fi"` et `production_countries` vaut `"United States of America, France"`. Les méthodes `.str` permettent de travailler sur ce texte.

**`str.contains` : est-ce que le texte contient un mot ?** Tu l'as déjà utilisé pour les genres. On s'en sert aussi pour repérer les films français :

```python
films["film_francais"] = films["production_countries"].str.contains("France", na=False)
print(films["film_francais"].sum(), "films français")
```

La colonne vaut Vrai ou Faux. `na=False` dit : si la case est vide (film absent de TMDB), on considère que ce n'est pas un film français. **La limite à écrire dans les slides :** une coproduction (par exemple un film américain coproduit par la France) compte comme française.

**`str.split` + `explode` : un genre par ligne.** Pour compter les films par genre, il faut découper la case.

```python
genres = films["genres"].str.split(",").explode()
print(genres.value_counts())
```

- `str.split(",")` transforme `"Action,Adventure"` en liste `["Action", "Adventure"]`.
- `explode()` crée une ligne par élément de la liste. Un film avec 3 genres donne 3 lignes.
- `value_counts()` compte combien de fois chaque genre apparaît (voir notion 8).

Attention : après `explode`, un film est compté dans plusieurs genres. Le total des genres est donc plus grand que le nombre de films, c'est normal.

<a id="notion8"></a>
## Notion 8 : agréger et résumer

Agréger, c'est résumer beaucoup de lignes en quelques chiffres : un total, une moyenne, une répartition. Ce sont ces chiffres qui alimentent le tableau de bord, et chacun doit répondre à une des 3 questions de la commission.

| Outil | Ce qu'il fait | Exemple |
| --- | --- | --- |
| `mean()`, `median()` | Moyenne, médiane | `films["averageRating"].mean()` |
| `quantile(0.9)` | Valeur sous laquelle se trouvent 90 % des films | `films["numVotes"].quantile(0.9)` |
| `value_counts()` | Compte chaque valeur d'une colonne | Nombre de films par genre, par décennie |
| `groupby(...).agg(...)` | Calcule un résumé par groupe | Note moyenne par décennie |

**Les KPI de ton tableau de bord, en code :**

```python
print(len(films), "films")
films["decennie"] = (films["startYear"] // 10) * 10
print(films["decennie"].value_counts().sort_index())
print(films["film_francais"].mean() * 100, "% de films français")

print(films["averageRating"].mean(), films["averageRating"].median())
print(films["numVotes"].median())

print(films.groupby("decennie")["averageRating"].mean())
```

- Les 4 premières lignes répondent à la question 1 (qu'est-ce que vous proposez ?), les 2 suivantes à la question 2 (est-ce que ce sont de bons films ?).
- `// 10` est la division entière : 1984 // 10 = 198, puis × 10 = 1980.
- `films["film_francais"].mean()` marche parce que Vrai vaut 1 et Faux vaut 0 : la moyenne donne directement la part de films français.
- `groupby("decennie")` fait des paquets de films par décennie, puis calcule la moyenne des notes dans chaque paquet.

<a id="notion9"></a>
## Notion 9 : la note pondérée (WR)

La note pondérée sert à classer les films à mettre à l'affiche (question 3 de la commission). Elle corrige un défaut de la note simple : un film noté 8,5 par 10 000 personnes est moins sûr qu'un film noté 8,3 par 1 million de personnes.

$$WR = \frac{v}{v+m} \times R + \frac{m}{v+m} \times C$$

- **R** : la note du film (`averageRating`).
- **v** : son nombre de votes (`numVotes`).
- **m** : un seuil de votes, par exemple le 90e percentile du catalogue.
- **C** : la note moyenne de tout le catalogue.

**L'idée en une phrase :** c'est un mélange entre la note du film et la note moyenne. Plus le film a de votes, plus on fait confiance à sa propre note. Avec peu de votes, sa note est « tirée » vers la moyenne.

```python
C = films["averageRating"].mean()
m = films["numVotes"].quantile(0.90)
v = films["numVotes"]
R = films["averageRating"]
films["WR"] = v / (v + m) * R + m / (v + m) * C

top = films.sort_values("WR", ascending=False)
print(top[["titre_fr", "averageRating", "numVotes", "WR"]].head(20))
```

Pour aller plus loin dans le tableau de bord : le top 10 par genre, ou le top 10 des films français. Ce sont de bonnes idées de « cycles » pour le cinéma.

<a id="notion10"></a>
## Notion 10 : visualisation et tableau de bord

Un graphique doit répondre à une question précise. Pour chaque question de la commission, voici le type de graphique qui convient :

| Question de la commission | Ce qu'on montre | Type de graphique |
| --- | --- | --- |
| Qu'est-ce que vous proposez ? | Nombre de films, films par genre, par décennie, part de films français | Carte KPI (un gros chiffre), barres, anneau |
| Est-ce que ce sont de bons films ? | Note moyenne et médiane, répartition des notes, votes médians | Carte KPI, histogramme |
| Qu'est-ce que vous mettez en avant ? | Top 10 selon la note pondérée, par genre | Tableau classé avec affiches, barres horizontales |

**Comment faire en pratique :**

1. Dans le notebook, teste rapidement avec pandas : `films["decennie"].value_counts().sort_index().plot(kind="bar")`. C'est pour vérifier que les chiffres ont du sens.
2. Utilise le fichier `data/catalogue_final.csv` produit par ce notebook.
3. Dans Power BI (ou Looker Studio), importe ce CSV et construis le vrai tableau de bord.
4. Pour les genres, il est plus simple d'exporter aussi un petit fichier « un genre par ligne » (notion 7), car Power BI ne sait pas découper `"Action,Comedy"` tout seul :

```python
genres = films[["tconst", "genres"]].assign(genre=films["genres"].str.split(",")).explode("genre")
genres[["tconst", "genre"]].to_csv(dossier_data / "films_genres.csv", index=False)
```

Dans Power BI, tu relies ensuite les deux fichiers par la colonne `tconst`.

<a id="notion11"></a>
## Notion 11 : préparer les données pour un modèle

Un modèle de recommandation ne comprend que des **nombres**. Il faut donc transformer chaque film en une ligne de nombres qui décrit ses caractéristiques. On appelle cette ligne un **vecteur**, et le tableau de tous les vecteurs s'appelle souvent `X`.

**Étape 1 : l'encodage des genres.** On transforme la case `"Action,Comedy"` en une colonne par genre, qui vaut 1 si le film a ce genre et 0 sinon.

```python
genres_01 = films["genres"].str.get_dummies(sep=",")
print(genres_01.head())
```

| Film | Action | Comedy | Horror | ... |
| --- | --- | --- | --- | --- |
| Alien | 0 | 0 | 1 | ... |
| Star Wars | 1 | 0 | 0 | ... |

**Étape 2 : la mise à l'échelle.** Les colonnes numériques n'ont pas la même taille : l'année vaut environ 2000, la note environ 7, les votes jusqu'à 3 millions. Sans correction, le modèle ne regarderait presque que les votes et l'année, parce que leurs chiffres sont les plus gros. `MinMaxScaler` ramène chaque colonne entre 0 et 1.

```python
import numpy as np
from sklearn.preprocessing import MinMaxScaler

num = films[["startYear", "runtimeMinutes", "averageRating"]].copy()
num["log_votes"] = np.log1p(films["numVotes"])
num_01 = MinMaxScaler().fit_transform(num)
```

`np.log1p` « écrase » les gros nombres de votes. Sans lui, quelques films très connus seraient à 1 et presque tous les autres près de 0.

**Étape 3 : coller les morceaux.**

```python
X = np.hstack([genres_01.values, num_01])
```

**Bonus :** `TfidfVectorizer` de scikit-learn transforme le résumé (`overview`) en nombres selon les mots importants. Deux films dont les résumés parlent d'espace et de vaisseau seront proches. Tu peux aussi donner plus de poids aux genres en les multipliant (par exemple `genres_01.values * 2`) : c'est un choix à justifier dans les slides.

<a id="notion12"></a>
## Notion 12 : la recommandation par similarité

Une fois chaque film transformé en vecteur (notion 11), recommander revient à chercher **les films dont les vecteurs se ressemblent le plus**. C'est une recommandation « basée sur le contenu » : elle n'a besoin d'aucun client, ce qui règle le problème du démarrage à froid.

**La similarité cosinus.** On compare deux vecteurs en regardant s'ils « pointent dans la même direction ». Deux films qui ont les mêmes genres, une époque et une note proches ont une distance cosinus proche de 0. Deux films qui n'ont rien en commun ont une distance proche de 1.

**Les plus proches voisins (`NearestNeighbors`).** Cet outil de scikit-learn trouve, pour un film donné, les N films les plus proches.

```python
from sklearn.neighbors import NearestNeighbors

films = films.reset_index(drop=True)
knn = NearestNeighbors(n_neighbors=6, metric="cosine")
knn.fit(X)

def recommander(titre):
    i = films.index[films["titre_fr"] == titre][0]
    distances, positions = knn.kneighbors(X[[i]])
    return films.iloc[positions[0][1:]]

print(recommander("Alien")[["titre_fr", "startYear", "genres"]])
```

- `reset_index(drop=True)` est important : après les filtres, les numéros de ligne de `films` ont des trous, alors que `X` est numéroté à partir de 0.
- `i` est la position du film aimé dans le tableau.
- `n_neighbors=6` donne 6 films : le film lui-même (il est toujours le plus proche de lui-même) et 5 recommandations.
- `positions[0][1:]` saute la première position, qui est le film choisi.

**Tester et justifier.** Essaie avec 5 films que tu connais bien et regarde si les résultats ont du sens. Si *Alien* recommande surtout des comédies, c'est que les poids sont mal réglés. Note ces essais : ils servent dans les slides pour expliquer le choix des colonnes.

<a id="notion13"></a>
## Notion 13 : Streamlit

Streamlit transforme un fichier Python en page web, sans écrire de HTML. Chaque fonction `st.quelquechose` ajoute un élément à la page : un titre, une liste déroulante, une image. À chaque clic de l'utilisateur, Streamlit relance le fichier de haut en bas.

| Fonction | Rôle dans ton appli |
| --- | --- |
| `st.title("Ciné Maniak")` | Le titre de la page |
| `st.selectbox("Un film que tu as aimé", liste)` | La liste déroulante pour choisir un film (avec les titres FR) |
| `@st.cache_data` | Charge le CSV une seule fois au lieu de le relire à chaque clic |
| `st.columns(5)` | Crée 5 colonnes côte à côte, une par film recommandé |
| `st.image(url, caption=titre)` | Affiche l'affiche avec le titre en dessous |

**Le squelette de `app/app.py` :**

```python
import streamlit as st
import pandas as pd

@st.cache_data
def charger():
    return pd.read_csv("data/catalogue_final.csv")

films = charger()
st.title("Ciné Maniak")
choix = st.selectbox("Un film que tu as aimé", sorted(films["titre_fr"]))

recos = recommander(choix)
colonnes = st.columns(5)
for col, (_, film) in zip(colonnes, recos.iterrows()):
    col.image(film["affiche"], caption=film["titre_fr"])
```

- `recommander` est la fonction de la notion 12.
- La colonne `affiche` contient déjà l'adresse complète de l'image (elle est créée dans la partie 5 de ce notebook).

Pour lancer l'appli, dans le terminal de VS Code : `streamlit run app/app.py`. Une page s'ouvre dans ton navigateur.

Pour que `recommander` fonctionne dans l'appli, il faut y recopier la préparation de `X` et le `NearestNeighbors` (notions 11 et 12), dans une fonction marquée `@st.cache_resource` pour ne les calculer qu'une fois. Pense aussi aux films sans affiche : affiche une image par défaut ou juste le titre.

<a id="notion14"></a>
## Notion 14 : Git et GitHub

Git garde l'historique de ton code, et GitHub le partage avec l'équipe. Tu travailles sur ta propre **branche**, `sebastien-preparation`, pour ne pas casser le travail des autres sur `main`.

**Le cycle à répéter après chaque avancée**, dans le terminal de VS Code, depuis le dossier `projet2-cinema` :

```bash
git status
git add notebooks/catalogue_final.ipynb
git commit -m "Ajout du notebook catalogue final"
git push
```

1. `git status` liste les fichiers modifiés.
2. `git add` choisit les fichiers à enregistrer.
3. `git commit -m "..."` enregistre une version avec un message qui dit ce que tu as fait.
4. `git push` envoie cette version sur GitHub.

**Pour partager avec l'équipe :** sur la page GitHub du dépôt, ouvre une *pull request* de `sebastien-preparation` vers `main`. Un coéquipier relit, puis fusionne.

**Ne jamais mettre dans git** le fichier TMDB (674 Mo) : GitHub refuse les fichiers de plus de 100 Mo. Vérifie que le fichier `.gitignore` contient bien `*.csv` ou le chemin du fichier TMDB.